# Result and corner-comparison ratings

Create ordinary training/reporting columns from the pinned Premier League 2024/25 publication,
then reuse saved rating states. See the [ratings guide](../docs/analytics/ratings.md).

Availability timestamps are optional. This example uses earlier kickoff as a retrospective proxy;
it does not establish the historical time when each result or statistic became available.

In [1]:
from pathlib import Path
import pandas as pd
from xdiyo_analytics.data import load_season, select_stats
from xdiyo_analytics.histories import build_team_history
from xdiyo_analytics.features import Stat, IsHome, MatchResultGlicko, StatGlicko, Rating, evaluate_features
from xdiyo_analytics.ratings import build_ratings, RatingRun

project_root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')

In [2]:
season = load_season(
    project_root / 'data/xDiyo_data', 'Premier_League_24_25', tables=['matches', 'statistics'],
    record_path=project_root / 'experiment/initial_population/selections/Premier_League_24_25.json',
)
selected = select_stats(season, stats=[('ALL', 'Match overview', 'cornerKicks')])
history = build_team_history(selected)
print(f'{len(season.matches)} matches; {len(history)} team rows')

380 matches; 760 team rows


## Choose rating features

Match-result ratings use W/D/L. Corner ratings compare corners earned with corners conceded:
higher/equal/lower becomes a win/draw/loss, independently of the match result.
They do not predict corner counts. Both sides are exported; default fields are public `rating`, `rd` and `sigma`.

Defaults are 1500 / 350 / .06 with tau=1, using the unchanged legacy numerical engine.
Replay updates per event, batching equal release times against shared prior states.
It inserts no calendar idle periods. These are explicit initial settings for this workflow.

In [3]:
corners = Stat('ALL', 'Match overview', 'cornerKicks')
features = evaluate_features(history, {
    'home': IsHome(), 'wdl': MatchResultGlicko(), 'corners': StatGlicko(corners),
})
print(features.shape)
pd.DataFrame({
    'team': history['team_name'], 'opponent': history['opponent_name'],
    'result_rating': features['wdl::result::team::rating'],
    'corner_rating': features['corners::ALL::Match overview::cornerKicks::value::team::rating'],
}).tail(4)

(760, 13)


,team,opponent,result_rating,corner_rating
756,Brighton & Hove Albion,Tottenham Hotspur,1575.804386,1510.059915
757,Tottenham Hotspur,Brighton & Hove Albion,1373.740798,1591.716335
758,Manchester City,Fulham,1626.433091,1612.826363
759,Fulham,Manchester City,1534.978801,1544.779051


## Save full state and choose model columns later

These version-pinned example runs are already saved by verification. The cell loads them;
in a new destination it builds and saves them first. Saving refuses existing artifacts.
Use a new definition/version directory and replay when history or rating settings change.

Snapshots retain public rating/RD/volatility and internal `mu`/`phi`, even when the model uses only two fields.
Future custom numeric states can use the same interface; no graph-model training is implemented here.

In [4]:
saved_root = (project_root / 'experiment/ratings_demo/Premier_League_24_25'
              / season.provenance['version'])
runs = {}
for name, statistic in [('result', None), ('corners', corners)]:
    directory = saved_root / name
    if (directory / 'ratings.json').exists():
        runs[name] = RatingRun.load(directory)
    else:
        runs[name] = build_ratings(history, stat=statistic)
        runs[name].save(directory)
print({name: len(run.snapshots) for name, run in runs.items()})
list(runs['result'].initial_state)

{'result': 760, 'corners': 760}


['rating', 'rd', 'sigma', 'mu', 'phi']

In [5]:
X = evaluate_features(history, {
    'home': IsHome(),
    'wdl': Rating('result', fields=('rating', 'rd')),
    'corners': Rating('corners', fields=('rating', 'rd')),
}, ratings=runs)
print(X.shape)
X.tail(4)

(760, 9)


,home,wdl::result::team::rating,wdl::result::team::rd,wdl::result::opponent::rating,wdl::result::opponent::rd,corners::ALL::Match overview::cornerKicks::value::team::rating,corners::ALL::Match overview::cornerKicks::value::team::rd,corners::ALL::Match overview::cornerKicks::value::opponent::rating,corners::ALL::Match overview::cornerKicks::value::opponent::rd
756,0.0,1575.804386,74.634622,1373.740798,75.354322,1510.059915,74.434600,1591.716335,79.313608
757,1.0,1373.740798,75.354322,1575.804386,74.634622,1591.716335,79.313608,1510.059915,74.434600
758,0.0,1626.433091,75.673913,1534.978801,74.795577,1612.826363,81.482965,1544.779051,74.975264
759,1.0,1534.978801,74.795577,1626.433091,75.673913,1544.779051,74.975264,1612.826363,81.482965
